In [1]:
import numpy as np
import pandas as pd
from datasets import load_dataset

ds1 = load_dataset("lavita/medical-qa-datasets", "all-processed")
ds2 = load_dataset("lavita/AlpaCare-MedInstruct-52k")
ds3 = load_dataset("lavita/MedREQAL")

df1 = ds1["train"].to_pandas()
df2 = ds2["train"].to_pandas()
df3 = ds3["train"].to_pandas()

print("Dataset1 columns:", df1.columns.tolist())
print("Dataset2 columns:", df2.columns.tolist())
print("Dataset3 columns:", df3.columns.tolist())
print("Counts:", len(df1), len(df2), len(df3))

columns = ["question", "background", "objective", "conclusion"]
df3 = df3[columns]
print("Dataset3 columns:", df3.columns.tolist())

Dataset1 columns: ['instruction', 'input', 'output', '__index_level_0__']
Dataset2 columns: ['output', 'input', 'instruction']
Dataset3 columns: ['question', 'background', 'objective', 'conclusion', 'verdicts', 'strength', 'label', 'category']
Counts: 239357 52002 2786
Dataset3 columns: ['question', 'background', 'objective', 'conclusion']


In [2]:
df3_mapped = pd.DataFrame()
df3_mapped["instruction"] = df3["question"].apply(lambda x: x.strip() if isinstance(x, str) else "")
df3_mapped["input"] = df3.apply(
    lambda row: "\n".join(
        filter(
            None,
            [
                row["background"].strip() if isinstance(row["background"], str) else "",
                row["objective"].strip() if isinstance(row["objective"], str) else "",
            ],
        )
    ),
    axis=1,
)
df3_mapped["output"] = df3["conclusion"].apply(lambda x: x.strip() if isinstance(x, str) else "")

In [3]:
#
# REMOVE QUOTATION MARKS + MERGE
#

df2 = df2.applymap(lambda x: x.strip('"').strip("'") if isinstance(x, str) else x)
df1 = df1.applymap(lambda x: x.strip('"').strip("'") if isinstance(x, str) else x)
df = pd.concat([df1, df2, df3_mapped], ignore_index=True)
df.drop("__index_level_0__", inplace=True, axis=1)

C:\Users\mati\AppData\Local\Temp\ipykernel_15384\905213411.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df2 = df2.applymap(lambda x: x.strip('"').strip("'") if isinstance(x, str) else x)
C:\Users\mati\AppData\Local\Temp\ipykernel_15384\905213411.py:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df1 = df1.applymap(lambda x: x.strip('"').strip("'") if isinstance(x, str) else x)


In [4]:
#
# REMOVE EMPTY ROWS
#


def removeEmpty(df):
    df["instruction"].replace("", np.nan, inplace=True)
    df["input"].replace("", np.nan, inplace=True)
    df["output"].replace("", np.nan, inplace=True)
    return df.dropna()


print(f"Before removal {len(df)}")
df_cleaned = removeEmpty(df)
print(f"After removal {len(df_cleaned)}")

pd.concat([df_cleaned, df]).drop_duplicates(keep=False)

Before removal 294145
After removal 293725


C:\Users\mati\AppData\Local\Temp\ipykernel_15384\1113083008.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["instruction"].replace("", np.nan, inplace=True)
C:\Users\mati\AppData\Local\Temp\ipykernel_15384\1113083008.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For ex

,instruction,input,output
4924,"If you are a doctor, please answer the medical...",NaN,"hello, 95% of the times' fever in this age gro..."
38944,"If you are a doctor, please answer the medical...",NaN,"hello, i have a seven day cycle. what day woul..."
74180,"If you are a doctor, please answer the medical...",NaN,"hi good day, welcome to chatbot. if your perio..."
77473,Answer this question truthfully,NaN,"I'm sorry, but the given answer ""Symmetric, we..."
80643,"If you are a doctor, please answer the medical...",NaN,hymen with an abnormal sperm count can conceiv...
134711,Answer this question truthfully,NaN,What is the name of the duct that drains the s...
135166,Answer this question truthfully,NaN,There is no vaccine to prevent Lassa fever. Pr...
155338,Answer this question truthfully,NaN,What is the name of the duct that drains the s...
163332,Answer this question truthfully,NaN,Lassa virus commonly involves the liver and re...
172949,"If you are a doctor, please answer the medical...",NaN,hellothanks for query. based n the facts that ...


In [5]:
df_cleaned[df_cleaned.isna().any(axis=1)]

,instruction,input,output


In [6]:
# #
# # REMOVE NON ENGLISH ROWS
# #

# def is_english(text):
#     try:
#         return detect(text) == "en"
#     except LangDetectException:
#         return False

# mask_non_english = ~df_cleaned["instruction"].apply(is_english)
# removed_non_english = df_cleaned[mask_non_english]

# df_final = df_cleaned[~mask_non_english]

# print("Rows removed (non-English):", len(removed_non_english))
# print(len(df_final))
# removed_non_english.head(50)

In [7]:
# #
# # REMOVE NON ENGLISH ROWS
# #

# from lingua import Language, LanguageDetectorBuilder

# detector = LanguageDetectorBuilder.from_languages(Language.ENGLISH).build()

# def is_english(text):
#     lang = detector.detect_language_of(text)
#     return lang == Language.ENGLISH

# mask_non_english = ~df_cleaned["instruction"].apply(is_english)
# removed_non_english = df_cleaned[mask_non_english]

# df_final = df_cleaned[~mask_non_english]

# print("Rows removed (non-English):", len(removed_non_english))
# print(len(df_final))

# from lingua import Language, LanguageDetectorBuilder

# detector = LanguageDetectorBuilder.from_languages(Language.ENGLISH).build()
# cols = ["instruction", "input", "output"]
# threshold = 0.6

# def row_is_english(row, min_length=50):
#     for c in cols:
#         val = row[c]
#         if isinstance(val, str) and len(val.strip()) >= min_length:
#             conf = detector.compute_language_confidence(val, Language.ENGLISH)
#             if conf < threshold:
#                 return False
#     return True

# mask_english = df_cleaned.apply(row_is_english, axis=1)
# df_final = df_cleaned[mask_english]
# removed_non_english = df_cleaned[~mask_english]

# print("Rows removed (non-English):", len(removed_non_english))
# print(len(df_final))
# removed_non_english.head(50)

In [8]:
#
# DROP DUPLICATES
#
df_final = df_cleaned.drop_duplicates(subset=["instruction", "output", "input"])
print(len(df_final))

293049


In [9]:
df_final[df_final.isna().any(axis=1)]

,instruction,input,output


In [10]:
import re


def clean_text(text):
    if pd.isna(text):
        return text
    text = re.sub(r"\([A-Z]{2,}\)", "", text)
    text = re.sub(r"\s{2,}", " ", text).strip()
    return text


df_final = df_final.applymap(clean_text)

# for col in ['instruction', 'input', 'output']:
#     df_final[col] = df_final[col].apply(clean_text)

C:\Users\mati\AppData\Local\Temp\ipykernel_15384\117633422.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_final = df_final.applymap(clean_text)


In [11]:
#
# Drop <noinput>
#


def is_noinput_variant(x):
    if not isinstance(x, str):
        return False
    s = re.sub(r"[^a-zA-Z]", "", x).lower()
    return sorted(s) == sorted("noinput")


if "input" in df_final.columns:
    df_final["input"] = df_final["input"].replace("<noinput>", "").str.strip()
print(len(df_final))


def clean_val(x):
    if not isinstance(x, str):
        return ""
    x = x.strip().strip('"').strip("'")
    if is_noinput_variant(x):
        return ""
    return x


# Final rows not removed
df_final = df_final.applymap(clean_val)
# Final rows removed
# df_fial_stripped = removeEmpty(df_final)

print(len(df_final))
# print(len(df_fial_stripped))

293049


C:\Users\mati\AppData\Local\Temp\ipykernel_15384\88277461.py:28: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_final = df_final.applymap(clean_val)


293049


In [ ]:
df_final.to_pickle("df_final.pkl")

In [1]:
import pandas as pd
df_final = pd.read_pickle("df_final.pkl")

In [ ]:
mask = df_final.apply(lambda c: c.astype(str).str.contains(r'\b\d+\s*stone\b', case=False, na=False))
mask_exclude = df_final.apply(lambda c: ~c.astype(str).str.contains(r'kidney', case=False, na=False))
final_mask = mask & mask_exclude

stone_units = df_final.where(final_mask)
matches = stone_units.stack()
texts = matches.values.tolist()

print(len(texts))
print("\n".join(texts))


205
Hello My blood pressure has always been completely normal until around 6 weeks ago, where it was 158/110. I am 42, 5 feet 0 high. At this time I weighed 10 1/2 stone, now 10 stone 2 lbs and bp is 132/92. Ive been power walking and working out for 1 1/2 hours per day 6 days per week for the past 6 weeks. My BP has dropped significantly, but the GP wants me to go on lifelong BP tablets - I have completely cut out salt from my diet also - do you think I can get down to normal BP by sustaining my current regime?
i have pain in upper abdomen, very sore to touch. chest pain also, after eating the chest pain is very bad and breathlessness. i have lost 3/4 stone in past 2 months as cannot eat much. had ct scan last wk, waiting on results but feel so weak and ill.
Hi I am 23 5 ft 7 and around 9 stone I have been suffering from stomach problems for around 6 months, in those 6 months my stomache has been in a lot of discomfort, I have been havin at least 4 bowel movements a day which i didnt 

In [ ]:
rows_to_drop = final_mask.any(axis=1)

df_cleaned = df_final[~rows_to_drop]

print(len(df_cleaned))


292844


In [ ]:
from generate_jsonl import generate_gemini_jsonl_batches_from_df

columns = ["instruction", "input", "output"]
generate_gemini_jsonl_batches_from_df(df_cleaned, "medical_df", columns, 30000)